# Multi-class Prediction of Obesity Risk competition

## Load and Prepare Data

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20758 entries, 0 to 20757
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              20758 non-null  int64  
 1   Gender                          20758 non-null  object 
 2   Age                             20758 non-null  float64
 3   Height                          20758 non-null  float64
 4   Weight                          20758 non-null  float64
 5   family_history_with_overweight  20758 non-null  object 
 6   FAVC                            20758 non-null  object 
 7   FCVC                            20758 non-null  float64
 8   NCP                             20758 non-null  float64
 9   CAEC                            20758 non-null  object 
 10  SMOKE                           20758 non-null  object 
 11  CH2O                            20758 non-null  float64
 12  SCC                             

,id,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,20758.00000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000,20758.000000
mean,10378.50000,23.841804,1.700245,87.887768,2.445908,2.761332,2.029418,0.981747,0.616756
std,5992.46278,5.688072,0.087312,26.379443,0.533218,0.705375,0.608467,0.838302,0.602113
min,0.00000,14.000000,1.450000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,5189.25000,20.000000,1.631856,66.000000,2.000000,3.000000,1.792022,0.008013,0.000000
50%,10378.50000,22.815416,1.700000,84.064875,2.393837,3.000000,2.000000,1.000000,0.573887
75%,15567.75000,26.000000,1.762887,111.600553,3.000000,3.000000,2.549617,1.587406,1.000000
max,20757.00000,61.000000,1.975663,165.057269,3.000000,4.000000,3.000000,3.000000,2.000000


### Encode Categorical Variables

In [2]:
from sklearn.preprocessing import LabelEncoder

y = train['NObeyesdad']
X = train.drop(columns=['NObeyesdad'])

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Align columns
X, test = X.align(test, join='left', axis=1, fill_value=0)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

## Model 1 — Multinomial Logistic Regression

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log', LogisticRegression(
        solver='lbfgs',
        max_iter=2000
    ))
])

log_pipeline.fit(X, y_encoded)

cv_log = cross_val_score(log_pipeline, X, y_encoded, cv=5, scoring='accuracy')
print("Logistic CV Accuracy:", cv_log.mean())

Logistic CV Accuracy: 0.8625595824113697


## Model 2 — LDA or QDA

In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X, y_encoded)

cv_lda = cross_val_score(lda_model, X, y_encoded, cv=5, scoring='accuracy')
print("LDA CV Accuracy:", cv_lda.mean())

LDA CV Accuracy: 0.8206959207081053


## Model 3 — Naïve Bayes

In [5]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X, y_encoded)

cv_nb = cross_val_score(nb_model, X, y_encoded, cv=5, scoring='accuracy')
print("Naive Bayes CV Accuracy:", cv_nb.mean())

Naive Bayes CV Accuracy: 0.6793524461222752


## Model 4 — Support Vector Machine

In [6]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf')
svm_model.fit(X, y_encoded)

cv_svm = cross_val_score(svm_model, X, y_encoded, cv=5, scoring='accuracy')
print("SVM CV Accuracy:", cv_svm.mean())

SVM CV Accuracy: 0.19491280277426942


## Submission Code

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# -------------------------
# 1. Load data
# -------------------------
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# -------------------------
# 2. Separate target
# -------------------------
y = train['NObeyesdad']
X = train.drop(columns=['NObeyesdad'])

# -------------------------
# 3. Encode categorical predictors
# -------------------------
X = pd.get_dummies(X, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Align train and test columns
X, test = X.align(test, join='left', axis=1, fill_value=0)

# -------------------------
# 4. Encode target variable
# -------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# -------------------------
# 5. Build SVM pipeline
# -------------------------
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

# -------------------------
# 6. Fit model
# -------------------------
svm_pipeline.fit(X, y_encoded)

# -------------------------
# 7. Predict test data
# -------------------------
test_predictions = svm_pipeline.predict(test)
test_predictions_labels = le.inverse_transform(test_predictions)

# -------------------------
# 8. Create submission file
# -------------------------
submission = pd.DataFrame({
    'id': test['id'],   # Use the actual ID column
    'NObeyesdad': test_predictions_labels
})

submission.to_csv('submission.csv', index=False)


Submission file created successfully!
